# DCN v2 + PLE-MTL — Regression-to-Mean (Amazon Video Games)

Colab notebook: bootstrap, 200k sample, **DCN v2** (cross + deep) and **3-task PLE** (general / low / high segments) with percentile or classifier routing. **Five rounds** (A–M) with JSON caches. Goal: **R² > 0.05 AND MAE < 0.85**.

Use `SKIP_ROUND_*` and cache JSON under `CACHE_DIR` to avoid re-running finished experiments.

In [ ]:
import os

if os.path.ismount('/content/drive'):
    print('Drive already mounted.')
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        print('Skipping drive mount (not in Colab UI or drive unavailable).')

In [ ]:
import os, subprocess

WORK_DIR = '/content/drive/MyDrive/colab/amazon_review_game'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
import os, subprocess

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = 'main'

if os.path.exists(os.path.join(repo_dir, '.git')):
    subprocess.run(['git', '-C', repo_dir, 'pull', 'origin', branch_name])
else:
    subprocess.run(['git', 'clone', repo_url])
    subprocess.run(['git', '-C', repo_dir, 'checkout', branch_name])

os.chdir(repo_dir)
print(f'Repo directory: {os.getcwd()}')

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'torch', 'pandas', 'numpy',
                'scikit-learn', 'matplotlib', 'seaborn', 'requests', 'scipy'])
try:
    subprocess.run(['nvidia-smi'], capture_output=True, check=False)
except FileNotFoundError:
    print('nvidia-smi not found (CPU/local env)')
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
_dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = torch.zeros(1, device=_dev)
print('Smoke tensor device:', x.device)

## GPU review (pre-execution checklist)

**Framework:** PyTorch  
**Device setup:** `device = cuda if available else cpu`; models `.to(device)`; batches `.to(device, non_blocking=pin_memory)`.  
**Overall:** OK for Colab GPU

### Findings
- **Critical:** None — no hardcoded CPU; training/eval use the same `device`.
- **Performance:** `DataLoader(..., pin_memory=True)` when CUDA; `torch.inference_mode()` on eval paths; `non_blocking=True` on host→device copies.

### Summary
Use a **GPU** Colab runtime; if `torch.cuda.is_available()` is False after install, switch runtime to GPU and re-run the install cell.

In [ ]:
# Config
import os

DATASET_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz'
PROJECT_NAME = 'amazon_review_game'
force_rewrite = False
SAMPLE_SIZE = 200_000

# Smoke test: set NOTEBOOK_SMOKE=1 locally for fast papermill (small data + few epochs)
SMOKE_TEST = os.environ.get('NOTEBOOK_SMOKE', '').strip() == '1'
if SMOKE_TEST:
    SAMPLE_SIZE = 5_000
    print('SMOKE_TEST=1: SAMPLE_SIZE=5000')

CACHE_DIR = '/content/drive/MyDrive/colab/data/experiment_cache/dcn_ple_rtm'
os.makedirs(CACHE_DIR, exist_ok=True)

SKIP_ROUND_1 = False
SKIP_ROUND_2 = False
SKIP_ROUND_3 = False
SKIP_ROUND_4 = False
SKIP_ROUND_5 = False

RTM_GOAL_R2 = 0.05
RTM_GOAL_MAE = 0.85

_epochs_main = 3 if SMOKE_TEST else 30
_epochs_router = 2 if SMOKE_TEST else 20
_epochs_long = 3 if SMOKE_TEST else 50
_epochs_vlong = 3 if SMOKE_TEST else 60

In [ ]:
# Download dataset
import os
import urllib.request

DATA_DIR = f'/content/drive/MyDrive/colab/data/{PROJECT_NAME}'
os.makedirs(DATA_DIR, exist_ok=True)

reviews_file = os.path.join(DATA_DIR, 'Video_Games.jsonl.gz')
should_download = force_rewrite or not os.path.exists(reviews_file) or os.path.getsize(reviews_file) == 0

if should_download:
    print(f'Downloading to {reviews_file}...')
    urllib.request.urlretrieve(DATASET_URL, reviews_file)
    print('Done.')
else:
    print(f'Using existing data at {reviews_file}')

In [ ]:
import gzip
import json
import pandas as pd

def load_jsonl_gz(path, max_rows=None):
    rows = []
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows:
                break
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return pd.DataFrame(rows)

df = load_jsonl_gz(reviews_file, max_rows=SAMPLE_SIZE)
print('Shape:', df.shape)
df.head()

In [ ]:
print('Task: regression (ratings 1–5)')

In [ ]:
# Preprocess
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

inter_df = df[['user_id', 'parent_asin', 'rating']].copy()
inter_df = inter_df.rename(columns={'parent_asin': 'item_id'})
inter_df = inter_df.dropna()
inter_df['rating'] = pd.to_numeric(inter_df['rating'], errors='coerce')
inter_df = inter_df.dropna()

user_enc = LabelEncoder()
item_enc = LabelEncoder()
inter_df['user_idx'] = user_enc.fit_transform(inter_df['user_id'].astype(str))
inter_df['item_idx'] = item_enc.fit_transform(inter_df['item_id'].astype(str))

n_users = int(inter_df['user_idx'].nunique())
n_items = int(inter_df['item_idx'].nunique())
print(f'Users: {n_users}, Items: {n_items}, Interactions: {len(inter_df)}')

train_df, test_df = train_test_split(inter_df, test_size=0.2, random_state=42)
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

In [ ]:
# Mean features (same as best prior: user_mean, item_mean)
global_mean = float(train_df['rating'].mean())
user_mean = train_df.groupby('user_idx')['rating'].mean()
item_mean = train_df.groupby('item_idx')['rating'].mean()

FEAT_COLS = ['user_mean', 'item_mean']

def build_features(df, user_mean, item_mean, global_mean, feat_cols=None):
    feat_cols = feat_cols or ['user_mean', 'item_mean']
    out = df.copy()
    out['user_mean'] = out['user_idx'].map(user_mean).fillna(global_mean)
    out['item_mean'] = out['item_idx'].map(item_mean).fillna(global_mean)
    return out

train_feat_df = build_features(train_df, user_mean, item_mean, global_mean, FEAT_COLS)
test_feat_df = build_features(test_df, user_mean, item_mean, global_mean, FEAT_COLS)
print('Features ready:', FEAT_COLS)

In [ ]:
##############################################################################
# Shared utilities: DCN v2, PLE-3task, routing, training, diagnostics
##############################################################################
import json
import time
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_pin = device.type == 'cuda'


def init_linear(layer: nn.Linear) -> None:
    nn.init.kaiming_uniform_(layer.weight, a=5 ** 0.5)
    if layer.bias is not None:
        nn.init.zeros_(layer.bias)


def compute_diagnostics(y_true, y_pred):
    sigma_true = float(np.std(y_true))
    sigma_pred = float(np.std(y_pred))
    ratio = sigma_pred / sigma_true if sigma_true > 0 else 0.0
    if len(y_pred) > 1 and np.var(y_pred) > 0:
        from numpy.polynomial import polynomial as P
        coefs = P.polyfit(y_pred, y_true, 1)
        calibration_slope = float(coefs[1])
    else:
        calibration_slope = 0.0
    return sigma_pred, sigma_true, ratio, calibration_slope


def metrics_dict(y_true, y_pred, label=''):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    sp, st, sr, cs = compute_diagnostics(y_true, y_pred)
    return {
        'label': label,
        'MAE': mae,
        'RMSE': rmse,
        'MSE': mse,
        'R2': r2,
        'sigma_ratio': sr,
        'cal_slope': cs,
    }


def print_metrics(m, elapsed=None):
    s = f"  >> MAE={m['MAE']:.4f}, RMSE={m['RMSE']:.4f}, R²={m['R2']:.4f} | σ_ratio={m['sigma_ratio']:.3f} cal_slope={m['cal_slope']:.3f}"
    if elapsed is not None:
        s += f" ({elapsed:.0f}s)"
    print(s)


# ── DCN v2 (stacked cross net + deep tower) ────────────────────────────────

class CrossNetV2(nn.Module):
    """DCN v2: x_{l+1} = x_0 ⊙ (W_l x_l + b_l) + x_l (full-rank)."""

    def __init__(self, d_in: int, num_layers: int = 3):
        super().__init__()
        self.d_in = d_in
        self.num_layers = num_layers
        self.linears = nn.ModuleList([nn.Linear(d_in, d_in) for _ in range(num_layers)])
        for lin in self.linears:
            init_linear(lin)

    def forward(self, x0: torch.Tensor) -> torch.Tensor:
        xl = x0
        for lin in self.linears:
            xl = x0 * lin(xl) + xl
        return xl


class DCNv2RecModel(nn.Module):
    def __init__(
        self,
        n_users: int,
        n_items: int,
        embed_dim: int = 32,
        n_extra_features: int = 2,
        num_cross_layers: int = 3,
        deep_hidden=(128, 64),
        dropout: float = 0.1,
        sigmoid_bound=(1.0, 5.0),
    ):
        super().__init__()
        self.user_emb = nn.Embedding(n_users + 1, embed_dim, padding_idx=0)
        self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        self.sigmoid_bound = sigmoid_bound
        d_in = embed_dim * 2 + n_extra_features
        self.cross = CrossNetV2(d_in, num_layers=num_cross_layers)
        layers = []
        h = d_in
        for hd in deep_hidden:
            layers += [nn.Linear(h, hd), nn.ReLU(), nn.Dropout(dropout)]
            h = hd
        self.deep = nn.Sequential(*layers)
        self.combine = nn.Linear(d_in + h, 1)
        init_linear(self.combine)

    def forward(self, user_idx, item_idx, extra_features=None):
        u = self.user_emb(user_idx + 1)
        i = self.item_emb(item_idx + 1)
        parts = [u, i]
        if extra_features is not None:
            parts.append(extra_features)
        x0 = torch.cat(parts, dim=1)
        xc = self.cross(x0)
        xd = self.deep(x0)
        logit = self.combine(torch.cat([xc, xd], dim=1)).squeeze(-1)
        lo, hi = self.sigmoid_bound
        return lo + (hi - lo) * torch.sigmoid(logit)


def train_feat_regressor(
    model,
    train_feat_df,
    test_feat_df,
    feat_cols,
    criterion,
    epochs=30,
    lr=1e-3,
    weight_decay=0.0,
    scheduler_type='cosine',
    label='',
):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = None
    if scheduler_type == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    X_u = torch.LongTensor(train_feat_df['user_idx'].values)
    X_i = torch.LongTensor(train_feat_df['item_idx'].values)
    X_f = torch.FloatTensor(train_feat_df[feat_cols].values)
    y = torch.FloatTensor(train_feat_df['rating'].values)
    loader = DataLoader(
        TensorDataset(X_u, X_i, X_f, y),
        batch_size=1024,
        shuffle=True,
        pin_memory=_pin,
    )

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        tot = 0.0
        for u, it, feat, r in loader:
            u = u.to(device, non_blocking=_pin)
            it = it.to(device, non_blocking=_pin)
            feat = feat.to(device, non_blocking=_pin)
            r = r.to(device, non_blocking=_pin)
            opt.zero_grad()
            pred = model(u, it, feat)
            loss = criterion(pred, r)
            loss.backward()
            opt.step()
            tot += loss.item()
        if scheduler:
            scheduler.step()
        if (epoch + 1) % max(1, epochs // 5) == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}/{epochs} loss={tot/len(loader):.4f}')
    elapsed = time.time() - t0

    model.eval()
    with torch.inference_mode():
        u_t = torch.LongTensor(test_feat_df['user_idx'].values).to(device)
        i_t = torch.LongTensor(test_feat_df['item_idx'].values).to(device)
        f_t = torch.FloatTensor(test_feat_df[feat_cols].values).to(device)
        preds = model(u_t, i_t, f_t).cpu().numpy()
    y_true = test_feat_df['rating'].values
    m = metrics_dict(y_true, preds, label=label)
    m['time'] = elapsed
    print_metrics(m, elapsed)
    return m


# ── PLE building blocks (3 tasks) ───────────────────────────────────────────

class ExpertMLP(nn.Module):
    def __init__(self, d_in: int, hidden: int, d_model: int, dropout: float = 0.0):
        super().__init__()
        self.fc1 = nn.Linear(d_in, hidden)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden, d_model)
        self.ln = nn.LayerNorm(d_model)
        init_linear(self.fc1)
        init_linear(self.fc2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        return self.ln(x)


class Gate(nn.Module):
    def __init__(self, d_selector: int, num_experts: int):
        super().__init__()
        self.linear = nn.Linear(d_selector, num_experts)
        init_linear(self.linear)

    def forward(self, selector: torch.Tensor) -> torch.Tensor:
        return torch.softmax(self.linear(selector), dim=-1)


class PLELevel3Task(nn.Module):
    """Shared + general + low + high expert groups; four gated fusion outputs."""

    def __init__(
        self,
        d_in: int,
        d_model: int,
        expert_hidden: int,
        num_shared: int,
        num_gen: int,
        num_low: int,
        num_high: int,
        dropout: float,
        d_sel_sh: int,
        d_sel_gen: int,
        d_sel_low: int,
        d_sel_high: int,
    ):
        super().__init__()
        Es, Eg, El, Eh = num_shared, num_gen, num_low, num_high
        self.shared = nn.ModuleList([ExpertMLP(d_in, expert_hidden, d_model, dropout) for _ in range(Es)])
        self.gen_e = nn.ModuleList([ExpertMLP(d_in, expert_hidden, d_model, dropout) for _ in range(Eg)])
        self.low_e = nn.ModuleList([ExpertMLP(d_in, expert_hidden, d_model, dropout) for _ in range(El)])
        self.high_e = nn.ModuleList([ExpertMLP(d_in, expert_hidden, d_model, dropout) for _ in range(Eh)])
        total = Es + Eg + El + Eh
        if total < 1:
            raise ValueError('Need at least one expert')
        self.g_sh = Gate(d_sel_sh, total)
        self.g_gen = Gate(d_sel_gen, total)
        self.g_low = Gate(d_sel_low, total)
        self.g_high = Gate(d_sel_high, total)

    def forward(self, x_exp, sel_sh, sel_gen, sel_low, sel_high):
        outs = []
        outs += [e(x_exp) for e in self.shared]
        outs += [e(x_exp) for e in self.gen_e]
        outs += [e(x_exp) for e in self.low_e]
        outs += [e(x_exp) for e in self.high_e]
        stacked = torch.stack(outs, dim=1)
        def mix(w):
            return (w.unsqueeze(-1) * stacked).sum(dim=1)
        return mix(self.g_sh(sel_sh)), mix(self.g_gen(sel_gen)), mix(self.g_low(sel_low)), mix(self.g_high(sel_high))


class TowerReg(nn.Module):
    def __init__(self, d_model: int, sigmoid_bound=(1.0, 5.0)):
        super().__init__()
        h = max(1, d_model // 2)
        self.fc1 = nn.Linear(d_model, h)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(h, 1)
        self.sigmoid_bound = sigmoid_bound
        init_linear(self.fc1)
        init_linear(self.fc2)

    def forward(self, x):
        z = self.act(self.fc1(x))
        logit = self.fc2(z).squeeze(-1)
        lo, hi = self.sigmoid_bound
        return lo + (hi - lo) * torch.sigmoid(logit)


class PLE3TaskRecModel(nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        embed_dim=32,
        n_extra_features=2,
        d_model=64,
        expert_hidden=128,
        num_shared=2,
        num_gen=1,
        num_low=1,
        num_high=1,
        dropout=0.1,
        sigmoid_bound=(1.0, 5.0),
    ):
        super().__init__()
        d_in = embed_dim * 2 + n_extra_features
        self.user_emb = nn.Embedding(n_users + 1, embed_dim, padding_idx=0)
        self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        self.level1 = PLELevel3Task(
            d_in, d_model, expert_hidden, num_shared, num_gen, num_low, num_high, dropout,
            d_sel_sh=d_in, d_sel_gen=d_in, d_sel_low=d_in, d_sel_high=d_in,
        )
        self.level2 = PLELevel3Task(
            d_in, d_model, expert_hidden, num_shared, num_gen, num_low, num_high, dropout,
            d_sel_sh=d_model, d_sel_gen=d_model, d_sel_low=d_model, d_sel_high=d_model,
        )
        self.t_gen = TowerReg(d_model, sigmoid_bound)
        self.t_low = TowerReg(d_model, sigmoid_bound)
        self.t_high = TowerReg(d_model, sigmoid_bound)

    def encode(self, user_idx, item_idx, extra_features):
        u = self.user_emb(user_idx + 1)
        i = self.item_emb(item_idx + 1)
        x = torch.cat([u, i, extra_features], dim=1)
        return x

    def forward_heads(self, x):
        s1, g1, l1, h1 = self.level1(x, x, x, x, x)
        s2, g2, l2, h2 = self.level2(x, s1, g1, l1, h1)
        pred_gen = self.t_gen(g2)
        pred_low = self.t_low(l2)
        pred_high = self.t_high(h2)
        return pred_gen, pred_low, pred_high

    def forward(self, user_idx, item_idx, extra_features=None):
        x = self.encode(user_idx, item_idx, extra_features)
        pg, pl, ph = self.forward_heads(x)
        return pg


def train_ple_mtl(
    model: PLE3TaskRecModel,
    train_feat_df,
    test_feat_df,
    feat_cols,
    epochs=30,
    lr=1e-3,
    w_gen=1.0,
    w_low=0.5,
    w_high=0.5,
    label='',
    low_rating_max=2.0,
    high_rating_min=4.0,
):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.SmoothL1Loss(reduction='none')

    X_u = torch.LongTensor(train_feat_df['user_idx'].values)
    X_i = torch.LongTensor(train_feat_df['item_idx'].values)
    X_f = torch.FloatTensor(train_feat_df[feat_cols].values)
    y = torch.FloatTensor(train_feat_df['rating'].values)
    loader = DataLoader(TensorDataset(X_u, X_i, X_f, y), batch_size=1024, shuffle=True, pin_memory=_pin)

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        tot = 0.0
        n_batches = 0
        for u, it, feat, r in loader:
            u = u.to(device, non_blocking=_pin)
            it = it.to(device, non_blocking=_pin)
            feat = feat.to(device, non_blocking=_pin)
            r = r.to(device, non_blocking=_pin)
            x = model.encode(u, it, feat)
            pg, pl, ph = model.forward_heads(x)
            mask_low = (r <= low_rating_max).float()
            mask_high = (r >= high_rating_min).float()
            lg = crit(pg, r).mean()
            ll = (crit(pl, r) * mask_low).sum() / (mask_low.sum().clamp_min(1.0))
            lh = (crit(ph, r) * mask_high).sum() / (mask_high.sum().clamp_min(1.0))
            loss = w_gen * lg + w_low * ll + w_high * lh
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item()
            n_batches += 1
        sched.step()
        if (epoch + 1) % max(1, epochs // 5) == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}/{epochs} loss={tot/max(n_batches,1):.4f}')
    elapsed = time.time() - t0
    return elapsed


@torch.inference_mode()
def ple_predict_all_heads(model: PLE3TaskRecModel, df, feat_cols):
    model.eval()
    u = torch.LongTensor(df['user_idx'].values).to(device)
    it = torch.LongTensor(df['item_idx'].values).to(device)
    f = torch.FloatTensor(df[feat_cols].values).to(device)
    x = model.encode(u, it, f)
    pg, pl, ph = model.forward_heads(x)
    return pg.cpu().numpy(), pl.cpu().numpy(), ph.cpu().numpy()


def compute_percentile_thresholds(train_feat_df, pred_gen, low_max=2.0, high_min=4.0):
    """P90 on general preds where true rating <= low_max; P10 where true rating >= high_min."""
    r = train_feat_df['rating'].values
    low_mask = r <= low_max
    high_mask = r >= high_min
    A = float(np.percentile(pred_gen[low_mask], 90)) if low_mask.any() else float('nan')
    B = float(np.percentile(pred_gen[high_mask], 10)) if high_mask.any() else float('nan')
    return A, B


def route_percentile(pred_gen, pred_low, pred_high, A, B):
    out = pred_gen.copy()
    if not np.isnan(A):
        m = pred_gen < A
        out[m] = pred_low[m]
    if not np.isnan(B):
        m2 = pred_gen > B
        out[m2] = pred_high[m2]
    return out


class ExpertDCN(nn.Module):
    """Expert = small DCN v2 stack (cross + deep) -> d_model."""

    def __init__(self, d_in, d_model, cross_layers=2, deep_hidden=(128,), dropout=0.1):
        super().__init__()
        self.cross = CrossNetV2(d_in, num_layers=cross_layers)
        layers = []
        h = d_in
        for hd in deep_hidden:
            layers += [nn.Linear(h, hd), nn.ReLU(), nn.Dropout(dropout)]
            h = hd
        self.deep = nn.Sequential(*layers)
        self.proj = nn.Linear(d_in + h, d_model)
        self.ln = nn.LayerNorm(d_model)
        init_linear(self.proj)

    def forward(self, x):
        xc = self.cross(x)
        xd = self.deep(x)
        z = self.ln(self.proj(torch.cat([xc, xd], dim=1)))
        return z


class PLELevel3TaskDCN(nn.Module):
    """PLE level with DCN-style experts (cross + deep)."""

    def __init__(
        self,
        d_in,
        d_model,
        expert_hidden,
        num_shared,
        num_gen,
        num_low,
        num_high,
        dropout,
        d_sel_sh,
        d_sel_gen,
        d_sel_low,
        d_sel_high,
        cross_layers=2,
        deep_hidden=(128,),
    ):
        super().__init__()
        Es, Eg, El, Eh = num_shared, num_gen, num_low, num_high
        mk = lambda: ExpertDCN(d_in, d_model, cross_layers, deep_hidden, dropout)
        self.shared = nn.ModuleList([mk() for _ in range(Es)])
        self.gen_e = nn.ModuleList([mk() for _ in range(Eg)])
        self.low_e = nn.ModuleList([mk() for _ in range(El)])
        self.high_e = nn.ModuleList([mk() for _ in range(Eh)])
        total = Es + Eg + El + Eh
        if total < 1:
            raise ValueError('Need at least one expert')
        self.g_sh = Gate(d_sel_sh, total)
        self.g_gen = Gate(d_sel_gen, total)
        self.g_low = Gate(d_sel_low, total)
        self.g_high = Gate(d_sel_high, total)

    def forward(self, x_exp, sel_sh, sel_gen, sel_low, sel_high):
        outs = [e(x_exp) for e in self.shared]
        outs += [e(x_exp) for e in self.gen_e]
        outs += [e(x_exp) for e in self.low_e]
        outs += [e(x_exp) for e in self.high_e]
        stacked = torch.stack(outs, dim=1)

        def mix(w):
            return (w.unsqueeze(-1) * stacked).sum(dim=1)

        return mix(self.g_sh(sel_sh)), mix(self.g_gen(sel_gen)), mix(self.g_low(sel_low)), mix(self.g_high(sel_high))


class PLE3TaskRecModelDCN(nn.Module):
    def __init__(
        self,
        n_users,
        n_items,
        embed_dim=32,
        n_extra_features=2,
        d_model=64,
        expert_hidden=128,
        num_shared=2,
        num_gen=1,
        num_low=1,
        num_high=1,
        dropout=0.1,
        sigmoid_bound=(1.0, 5.0),
        cross_layers=2,
        deep_hidden=(96,),
    ):
        super().__init__()
        d_in = embed_dim * 2 + n_extra_features
        self.user_emb = nn.Embedding(n_users + 1, embed_dim, padding_idx=0)
        self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        self.level1 = PLELevel3TaskDCN(
            d_in,
            d_model,
            expert_hidden,
            num_shared,
            num_gen,
            num_low,
            num_high,
            dropout,
            d_in,
            d_in,
            d_in,
            d_in,
            cross_layers=cross_layers,
            deep_hidden=deep_hidden,
        )
        self.level2 = PLELevel3TaskDCN(
            d_in,
            d_model,
            expert_hidden,
            num_shared,
            num_gen,
            num_low,
            num_high,
            dropout,
            d_model,
            d_model,
            d_model,
            d_model,
            cross_layers=cross_layers,
            deep_hidden=deep_hidden,
        )
        self.t_gen = TowerReg(d_model, sigmoid_bound)
        self.t_low = TowerReg(d_model, sigmoid_bound)
        self.t_high = TowerReg(d_model, sigmoid_bound)

    def encode(self, user_idx, item_idx, extra_features):
        u = self.user_emb(user_idx + 1)
        i = self.item_emb(item_idx + 1)
        return torch.cat([u, i, extra_features], dim=1)

    def forward_heads(self, x):
        s1, g1, l1, h1 = self.level1(x, x, x, x, x)
        s2, g2, l2, h2 = self.level2(x, s1, g1, l1, h1)
        return self.t_gen(g2), self.t_low(l2), self.t_high(h2)

    def forward(self, user_idx, item_idx, extra_features=None):
        x = self.encode(user_idx, item_idx, extra_features)
        pg, _, _ = self.forward_heads(x)
        return pg


class RouterMLP(nn.Module):
    """3-class router: 0=low (1–2), 1=mid (3), 2=high (4–5)."""

    def __init__(self, n_users, n_items, embed_dim=32, n_extra=2, hidden=(64, 32), dropout=0.1):
        super().__init__()
        self.user_emb = nn.Embedding(n_users + 1, embed_dim, padding_idx=0)
        self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        layers = []
        d = embed_dim * 2 + n_extra
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers.append(nn.Linear(d, 3))
        self.net = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                init_linear(m)

    def forward(self, u, i, feat):
        x = torch.cat([self.user_emb(u + 1), self.item_emb(i + 1), feat], dim=1)
        return self.net(x)


def rating_to_router_class(ratings: torch.Tensor) -> torch.Tensor:
    y = ratings.long()
    out = torch.full_like(y, 1)
    out[y <= 2] = 0
    out[y >= 4] = 2
    return out


def train_router(router, train_feat_df, feat_cols, epochs=20, lr=1e-3):
    router = router.to(device)
    opt = torch.optim.Adam(router.parameters(), lr=lr, weight_decay=1e-5)
    ce = nn.CrossEntropyLoss()
    X_u = torch.LongTensor(train_feat_df['user_idx'].values)
    X_i = torch.LongTensor(train_feat_df['item_idx'].values)
    X_f = torch.FloatTensor(train_feat_df[feat_cols].values)
    yc = rating_to_router_class(torch.FloatTensor(train_feat_df['rating'].values))
    loader = DataLoader(TensorDataset(X_u, X_i, X_f, yc), batch_size=1024, shuffle=True, pin_memory=_pin)
    for epoch in range(epochs):
        router.train()
        for u, it, feat, c in loader:
            u = u.to(device, non_blocking=_pin)
            it = it.to(device, non_blocking=_pin)
            feat = feat.to(device, non_blocking=_pin)
            c = c.to(device, non_blocking=_pin)
            opt.zero_grad()
            loss = ce(router(u, it, feat), c)
            loss.backward()
            opt.step()
    return router


@torch.inference_mode()
def route_classifier_preds(router, df, feat_cols, pg, pl, ph):
    router.eval()
    u = torch.LongTensor(df['user_idx'].values).to(device)
    it = torch.LongTensor(df['item_idx'].values).to(device)
    f = torch.FloatTensor(df[feat_cols].values).to(device)
    cls = router(u, it, f).argmax(dim=1).cpu().numpy()
    out = np.where(cls == 0, pl, np.where(cls == 2, ph, pg))
    return out


def print_round_summary(rows, round_num, ref_mae):
    print(f"\n{'='*60}")
    print(f'ROUND {round_num} SUMMARY (ref MAE={ref_mae:.4f})')
    print(f"{'='*60}")
    print(f"{'Experiment':<42} {'MAE':>7} {'R²':>7} {'σ_ratio':>7}")
    print('-' * 60)
    for r in rows:
        print(f"{r['label']:<42} {r['MAE']:>7.4f} {r['R2']:>7.4f} {r['sigma_ratio']:>7.3f}")


print(f'Shared utilities loaded. device={device} pin_memory={_pin}')

In [ ]:
##############################################################################
# ROUND 1: A DCN v2 | B PLE + percentile | C PLE + classifier routing
##############################################################################
import json
import os

_R1 = os.path.join(CACHE_DIR, 'round_1_results.json')

if not SKIP_ROUND_1 and not os.path.exists(_R1):
    round1 = []
    ref_mae = 0.874

    print('=' * 60)
    print('EXPERIMENT A: DCN v2 + mean feats + Huber + sigmoid [1,5]')
    print('=' * 60)
    model_a = DCNv2RecModel(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        num_cross_layers=3, deep_hidden=(128, 64), dropout=0.1,
    )
    ra = train_feat_regressor(
        model_a, train_feat_df, test_feat_df, FEAT_COLS,
        criterion=nn.SmoothL1Loss(), epochs=_epochs_main, lr=1e-3,
        scheduler_type='cosine', label='A: DCNv2+Huber',
    )
    round1.append(ra)

    print('=' * 60)
    print('EXPERIMENT B: PLE 3-task + percentile routing (train thresholds)')
    print('=' * 60)
    model_b = PLE3TaskRecModel(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        d_model=64, expert_hidden=128, num_shared=2, num_gen=1, num_low=1, num_high=1,
        dropout=0.1,
    )
    tb = train_ple_mtl(model_b, train_feat_df, test_feat_df, FEAT_COLS, epochs=_epochs_main, label='B')
    pg_tr, pl_tr, ph_tr = ple_predict_all_heads(model_b, train_feat_df, FEAT_COLS)
    A, B = compute_percentile_thresholds(train_feat_df, pg_tr)
    print(f'  Percentile thresholds: A (P90 gen | y<=2)={A:.4f}, B (P10 gen | y>=4)={B:.4f}')
    pg_te, pl_te, ph_te = ple_predict_all_heads(model_b, test_feat_df, FEAT_COLS)
    routed = route_percentile(pg_te, pl_te, ph_te, A, B)
    y_true = test_feat_df['rating'].values
    rb = metrics_dict(y_true, routed, label='B: PLE+percentile route')
    rb['time'] = tb
    print_metrics(rb, tb)
    round1.append(rb)

    print('=' * 60)
    print('EXPERIMENT C: PLE 3-task + router (3-class)')
    print('=' * 60)
    model_c = PLE3TaskRecModel(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        d_model=64, expert_hidden=128, num_shared=2, num_gen=1, num_low=1, num_high=1,
        dropout=0.1,
    )
    tc = train_ple_mtl(model_c, train_feat_df, test_feat_df, FEAT_COLS, epochs=_epochs_main, label='C')
    router_c = RouterMLP(n_users, n_items, embed_dim=32, n_extra=len(FEAT_COLS))
    train_router(router_c, train_feat_df, FEAT_COLS, epochs=_epochs_router)
    pg_te, pl_te, ph_te = ple_predict_all_heads(model_c, test_feat_df, FEAT_COLS)
    routed_c = route_classifier_preds(router_c, test_feat_df, FEAT_COLS, pg_te, pl_te, ph_te)
    rc = metrics_dict(y_true, routed_c, label='C: PLE+router')
    rc['time'] = tc
    print_metrics(rc, tc)
    round1.append(rc)

    def slim(d):
        return {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in d.items() if k in ('label', 'MAE', 'RMSE', 'R2', 'sigma_ratio', 'cal_slope', 'time')}

    with open(_R1, 'w') as fp:
        json.dump([slim(x) for x in round1], fp, indent=2)
    print_round_summary(round1, 1, ref_mae)
else:
    if os.path.exists(_R1):
        with open(_R1) as fp:
            round1 = json.load(fp)
        print('ROUND 1: SKIPPED (cached)')
        for r in round1:
            print(f"  {r['label']}: MAE={r['MAE']:.4f} R²={r['R2']:.4f}")
    else:
        print('ROUND 1: skipped (SKIP_ROUND_1=True and no cache)')
        round1 = []

In [ ]:
##############################################################################
# ROUND 2: capacity / loss-weight variants (optional)
##############################################################################
import json
import os

_R2 = os.path.join(CACHE_DIR, 'round_2_results.json')

if not SKIP_ROUND_2 and not os.path.exists(_R2):
    round2 = []
    ref_mae = 0.874

    print('=' * 60)
    print('EXPERIMENT D: DCN v2 deeper cross (5 layers)')
    print('=' * 60)
    model_d = DCNv2RecModel(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        num_cross_layers=5, deep_hidden=(128, 64), dropout=0.1,
    )
    rd = train_feat_regressor(
        model_d, train_feat_df, test_feat_df, FEAT_COLS,
        criterion=nn.SmoothL1Loss(), epochs=_epochs_main, lr=1e-3,
        scheduler_type='cosine', label='D: DCNv2 x5',
    )
    round2.append(rd)

    print('=' * 60)
    print('EXPERIMENT E: PLE + percentile, higher task weights')
    print('=' * 60)
    model_e = PLE3TaskRecModel(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        d_model=64, expert_hidden=128, num_shared=2, num_gen=1, num_low=1, num_high=1,
        dropout=0.1,
    )
    te = train_ple_mtl(
        model_e, train_feat_df, test_feat_df, FEAT_COLS,
        epochs=_epochs_main, w_gen=1.0, w_low=1.0, w_high=1.0, label='E',
    )
    pg_tr, _, _ = ple_predict_all_heads(model_e, train_feat_df, FEAT_COLS)
    A, B = compute_percentile_thresholds(train_feat_df, pg_tr)
    pg_te, pl_te, ph_te = ple_predict_all_heads(model_e, test_feat_df, FEAT_COLS)
    routed = route_percentile(pg_te, pl_te, ph_te, A, B)
    y_true = test_feat_df['rating'].values
    re = metrics_dict(y_true, routed, label='E: PLE+perc w=1')
    re['time'] = te
    print_metrics(re, te)
    round2.append(re)

    print('=' * 60)
    print('EXPERIMENT F: PLE wider (d_model=96) + percentile')
    print('=' * 60)
    model_f = PLE3TaskRecModel(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        d_model=96, expert_hidden=192, num_shared=2, num_gen=1, num_low=1, num_high=1,
        dropout=0.1,
    )
    tf_ = train_ple_mtl(model_f, train_feat_df, test_feat_df, FEAT_COLS, epochs=_epochs_main, label='F')
    pg_tr, _, _ = ple_predict_all_heads(model_f, train_feat_df, FEAT_COLS)
    A, B = compute_percentile_thresholds(train_feat_df, pg_tr)
    pg_te, pl_te, ph_te = ple_predict_all_heads(model_f, test_feat_df, FEAT_COLS)
    routed = route_percentile(pg_te, pl_te, ph_te, A, B)
    rf = metrics_dict(y_true, routed, label='F: PLE d96+perc')
    rf['time'] = tf_
    print_metrics(rf, tf_)
    round2.append(rf)

    def slim(d):
        return {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in d.items() if k in ('label', 'MAE', 'RMSE', 'R2', 'sigma_ratio', 'cal_slope', 'time')}

    with open(_R2, 'w') as fp:
        json.dump([slim(x) for x in round2], fp, indent=2)
    print_round_summary(round2, 2, ref_mae)
else:
    if os.path.exists(_R2):
        with open(_R2) as fp:
            round2 = json.load(fp)
        print('ROUND 2: SKIPPED (cached)')
        for r in round2:
            print(f"  {r['label']}: MAE={r['MAE']:.4f}")
    else:
        print('ROUND 2: skipped (SKIP_ROUND_2=True or no cache)')
        round2 = []

In [ ]:
##############################################################################
# ROUND 3: DCN backbone inside experts (optional)
##############################################################################
import json
import os

_R3 = os.path.join(CACHE_DIR, 'round_3_results.json')

if not SKIP_ROUND_3 and not os.path.exists(_R3):
    round3 = []
    ref_mae = 0.874

    print('=' * 60)
    print('EXPERIMENT G: PLE with DCN experts + percentile routing')
    print('=' * 60)
    model_g = PLE3TaskRecModelDCN(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        d_model=64, expert_hidden=128, num_shared=2, num_gen=1, num_low=1, num_high=1,
        dropout=0.1, cross_layers=2, deep_hidden=(96,),
    )
    tg = train_ple_mtl(model_g, train_feat_df, test_feat_df, FEAT_COLS, epochs=_epochs_main, label='G')
    pg_tr, _, _ = ple_predict_all_heads(model_g, train_feat_df, FEAT_COLS)
    A, B = compute_percentile_thresholds(train_feat_df, pg_tr)
    pg_te, pl_te, ph_te = ple_predict_all_heads(model_g, test_feat_df, FEAT_COLS)
    y_true = test_feat_df['rating'].values
    routed = route_percentile(pg_te, pl_te, ph_te, A, B)
    rg = metrics_dict(y_true, routed, label='G: PLE-DCN+perc')
    rg['time'] = tg
    print_metrics(rg, tg)
    round3.append(rg)

    def slim(d):
        return {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in d.items() if k in ('label', 'MAE', 'RMSE', 'R2', 'sigma_ratio', 'cal_slope', 'time')}

    with open(_R3, 'w') as fp:
        json.dump([slim(x) for x in round3], fp, indent=2)
    print_round_summary(round3, 3, ref_mae)
else:
    if os.path.exists(_R3):
        with open(_R3) as fp:
            round3 = json.load(fp)
        print('ROUND 3: SKIPPED (cached)')
        for r in round3:
            print(f"  {r['label']}: MAE={r['MAE']:.4f}")
    else:
        print('ROUND 3: skipped (SKIP_ROUND_3=True or no cache)')
        round3 = []

In [ ]:
##############################################################################
# ROUND 4: H,I,J — longer training, extreme tail masks, DCN patient
##############################################################################
import json
import os

_R4 = os.path.join(CACHE_DIR, 'round_4_results.json')

if not SKIP_ROUND_4 and not os.path.exists(_R4):
    round4 = []
    ref_mae = 0.874
    y_true = test_feat_df['rating'].values

    print('=' * 60)
    print('EXPERIMENT H: PLE + router, 50ep, lr=5e-4 (patient)')
    print('=' * 60)
    model_h = PLE3TaskRecModel(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        d_model=64, expert_hidden=128, num_shared=2, num_gen=1, num_low=1, num_high=1,
        dropout=0.1,
    )
    th = train_ple_mtl(
        model_h, train_feat_df, test_feat_df, FEAT_COLS,
        epochs=_epochs_long, lr=5e-4, label='H',
    )
    router_h = RouterMLP(n_users, n_items, embed_dim=32, n_extra=len(FEAT_COLS))
    train_router(router_h, train_feat_df, FEAT_COLS, epochs=_epochs_router, lr=5e-4)
    pg_te, pl_te, ph_te = ple_predict_all_heads(model_h, test_feat_df, FEAT_COLS)
    rh = metrics_dict(y_true, route_classifier_preds(router_h, test_feat_df, FEAT_COLS, pg_te, pl_te, ph_te), label='H: PLE+router 50ep lr5e-4')
    rh['time'] = th
    print_metrics(rh, th)
    round4.append(rh)

    print('=' * 60)
    print('EXPERIMENT I: DCN v2 + Huber, 50ep, lr=5e-4')
    print('=' * 60)
    model_i = DCNv2RecModel(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        num_cross_layers=3, deep_hidden=(128, 64), dropout=0.1,
    )
    ri = train_feat_regressor(
        model_i, train_feat_df, test_feat_df, FEAT_COLS,
        criterion=nn.SmoothL1Loss(), epochs=_epochs_long, lr=5e-4,
        scheduler_type='cosine', label='I: DCNv2 50ep lr5e-4',
    )
    round4.append(ri)

    print('=' * 60)
    print('EXPERIMENT J: PLE MTL masks only rating 1 / 5 + router')
    print('=' * 60)
    model_j = PLE3TaskRecModel(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        d_model=64, expert_hidden=128, num_shared=2, num_gen=1, num_low=1, num_high=1,
        dropout=0.1,
    )
    tj = train_ple_mtl(
        model_j, train_feat_df, test_feat_df, FEAT_COLS,
        epochs=_epochs_main, low_rating_max=1.0, high_rating_min=5.0, label='J',
    )
    router_j = RouterMLP(n_users, n_items, embed_dim=32, n_extra=len(FEAT_COLS))
    train_router(router_j, train_feat_df, FEAT_COLS, epochs=_epochs_router)
    pg_te, pl_te, ph_te = ple_predict_all_heads(model_j, test_feat_df, FEAT_COLS)
    rj = metrics_dict(y_true, route_classifier_preds(router_j, test_feat_df, FEAT_COLS, pg_te, pl_te, ph_te), label='J: PLE mask y=1/5+router')
    rj['time'] = tj
    print_metrics(rj, tj)
    round4.append(rj)

    def slim(d):
        return {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in d.items() if k in ('label', 'MAE', 'RMSE', 'R2', 'sigma_ratio', 'cal_slope', 'time')}

    with open(_R4, 'w') as fp:
        json.dump([slim(x) for x in round4], fp, indent=2)
    print_round_summary(round4, 4, ref_mae)
else:
    if os.path.exists(_R4):
        with open(_R4) as fp:
            round4 = json.load(fp)
        print('ROUND 4: SKIPPED (cached)')
        for r in round4:
            print(f"  {r['label']}: MAE={r['MAE']:.4f}")
    else:
        print('ROUND 4: skipped (SKIP_ROUND_4=True or no cache)')
        round4 = []

In [ ]:
##############################################################################
# ROUND 5: K,L,M — strict percentiles, wide PLE, PLE-DCN + router
##############################################################################
import json
import os

_R5 = os.path.join(CACHE_DIR, 'round_5_results.json')

if not SKIP_ROUND_5 and not os.path.exists(_R5):
    round5 = []
    ref_mae = 0.874
    y_true = test_feat_df['rating'].values

    print('=' * 60)
    print('EXPERIMENT K: PLE + percentile thresholds from y=1 and y=5 only')
    print('=' * 60)
    model_k = PLE3TaskRecModel(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        d_model=64, expert_hidden=128, num_shared=2, num_gen=1, num_low=1, num_high=1,
        dropout=0.1,
    )
    tk = train_ple_mtl(model_k, train_feat_df, test_feat_df, FEAT_COLS, epochs=_epochs_main, label='K')
    pg_tr, _, _ = ple_predict_all_heads(model_k, train_feat_df, FEAT_COLS)
    A, B = compute_percentile_thresholds(train_feat_df, pg_tr, low_max=1.0, high_min=5.0)
    print(f'  Thresholds (P90 gen|y=1, P10 gen|y=5): A={A:.4f}, B={B:.4f}')
    pg_te, pl_te, ph_te = ple_predict_all_heads(model_k, test_feat_df, FEAT_COLS)
    rk = metrics_dict(y_true, route_percentile(pg_te, pl_te, ph_te, A, B), label='K: PLE+perc thr(y=1,y=5)')
    rk['time'] = tk
    print_metrics(rk, tk)
    round5.append(rk)

    print('=' * 60)
    print('EXPERIMENT L: Wider PLE (d_model=80, 3 shared + 2 gen experts) + router')
    print('=' * 60)
    model_l = PLE3TaskRecModel(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        d_model=80, expert_hidden=160, num_shared=3, num_gen=2, num_low=1, num_high=1,
        dropout=0.1,
    )
    tl = train_ple_mtl(model_l, train_feat_df, test_feat_df, FEAT_COLS, epochs=_epochs_main, label='L')
    router_l = RouterMLP(n_users, n_items, embed_dim=32, n_extra=len(FEAT_COLS), hidden=(96, 48))
    train_router(router_l, train_feat_df, FEAT_COLS, epochs=_epochs_router)
    pg_te, pl_te, ph_te = ple_predict_all_heads(model_l, test_feat_df, FEAT_COLS)
    rl = metrics_dict(y_true, route_classifier_preds(router_l, test_feat_df, FEAT_COLS, pg_te, pl_te, ph_te), label='L: PLE wide+router')
    rl['time'] = tl
    print_metrics(rl, tl)
    round5.append(rl)

    print('=' * 60)
    print('EXPERIMENT M: PLE-DCN experts + router (same protocol as C)')
    print('=' * 60)
    model_m = PLE3TaskRecModelDCN(
        n_users, n_items, embed_dim=32, n_extra_features=len(FEAT_COLS),
        d_model=64, expert_hidden=128, num_shared=2, num_gen=1, num_low=1, num_high=1,
        dropout=0.1, cross_layers=2, deep_hidden=(96,),
    )
    tm = train_ple_mtl(model_m, train_feat_df, test_feat_df, FEAT_COLS, epochs=_epochs_main, label='M')
    router_m = RouterMLP(n_users, n_items, embed_dim=32, n_extra=len(FEAT_COLS))
    train_router(router_m, train_feat_df, FEAT_COLS, epochs=_epochs_router)
    pg_te, pl_te, ph_te = ple_predict_all_heads(model_m, test_feat_df, FEAT_COLS)
    rm = metrics_dict(y_true, route_classifier_preds(router_m, test_feat_df, FEAT_COLS, pg_te, pl_te, ph_te), label='M: PLE-DCN+router')
    rm['time'] = tm
    print_metrics(rm, tm)
    round5.append(rm)

    def slim(d):
        return {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in d.items() if k in ('label', 'MAE', 'RMSE', 'R2', 'sigma_ratio', 'cal_slope', 'time')}

    with open(_R5, 'w') as fp:
        json.dump([slim(x) for x in round5], fp, indent=2)
    print_round_summary(round5, 5, ref_mae)
else:
    if os.path.exists(_R5):
        with open(_R5) as fp:
            round5 = json.load(fp)
        print('ROUND 5: SKIPPED (cached)')
        for r in round5:
            print(f"  {r['label']}: MAE={r['MAE']:.4f}")
    else:
        print('ROUND 5: skipped (SKIP_ROUND_5=True or no cache)')
        round5 = []

In [ ]:
# Goal check + leaderboard (rounds 1–5)
import json
import os

all_rows = []
for path, rnd in [
    (os.path.join(CACHE_DIR, 'round_1_results.json'), 1),
    (os.path.join(CACHE_DIR, 'round_2_results.json'), 2),
    (os.path.join(CACHE_DIR, 'round_3_results.json'), 3),
    (os.path.join(CACHE_DIR, 'round_4_results.json'), 4),
    (os.path.join(CACHE_DIR, 'round_5_results.json'), 5),
]:
    if os.path.exists(path):
        with open(path) as fp:
            for r in json.load(fp):
                r = dict(r)
                r['round'] = rnd
                all_rows.append(r)

all_rows.sort(key=lambda x: x['MAE'])
print('\nOVERALL RESULTS (by MAE)')
print('=' * 70)
print(f"{'Exp':<36} {'Round':>5} {'MAE':>8} {'R²':>8} {'σ_ratio':>8}")
print('-' * 70)
for r in all_rows:
    print(f"{r['label'][:36]:<36} {r['round']:>5} {r['MAE']:>8.4f} {r['R2']:>8.4f} {r['sigma_ratio']:>8.3f}")

if all_rows:
    best = all_rows[0]
    met = (best['R2'] > RTM_GOAL_R2) and (best['MAE'] < RTM_GOAL_MAE)
    print(f"\nBest: {best['label']} | MAE={best['MAE']:.4f} R²={best['R2']:.4f}")
    print(f"Goal R²>{RTM_GOAL_R2} and MAE<{RTM_GOAL_MAE}: {'MET' if met else 'NOT MET'}")
else:
    print('No cached results yet. Run rounds with SKIP_ROUND_*=False.')